In [ ]:
# --- setup -------------------------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

REPO = '/content/pxr-repo'
!git clone -q https://github.com/pridem755/patient-or-xray.git $REPO 2>/dev/null || (cd $REPO && git pull -q)
%pip install -q -e $REPO

In [ ]:
import importlib
import site
import sys

site.main()
importlib.invalidate_caches()
SRC = f'{REPO}/src'
if SRC not in sys.path:
    sys.path.insert(0, SRC)

import pxr
print('pxr loaded from:', pxr.__file__)

In [ ]:
import shutil
from pathlib import Path

import pandas as pd

from pxr.config import load_config
from pxr.data.cache import build_cache, load_cache, verify_cache

cfg = load_config(f'{REPO}/config/study_config.yaml')
ROOT = Path(cfg.paths['drive_root'])
COHORTS = ROOT / cfg.paths['cohorts']
ZIPS = ROOT / cfg.paths['raw_zips']
CACHE = ROOT / cfg.paths['image_cache']
SCRATCH = Path(cfg.paths['local_scratch'])
SCRATCH.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)

print('config_hash :', cfg.config_hash)
print('image size :', cfg.model['image_size'])
print('zips :', ZIPS)
print('cache :', CACHE)

In [ ]:
# --- load cohorts ----------------------------------------------------------------
cohorts = {}
for site in cfg.site_names:
    cohorts[site] = pd.read_parquet(COHORTS / cfg.artifact_name('cohort', site=site))
    print(f'{site:<10} {len(cohorts[site]):>7,} images to cache')

total = sum(len(df) for df in cohorts.values())
size = cfg.model['image_size']
print(f'\ntotal {total:,} images -> about {total * size * size / 1e9:.1f} GB')

## 2. Locate the archives

Name each site's zip. The build fails loudly if a cohort image is absent from its
archive, rather than quietly caching a smaller set.

In [ ]:
# --- check for raw zip files ------------------------------------------------------
ARCHIVES = {
    'mimic-cxr': ZIPS / 'mimic_images.zip',
    'chexpert': ZIPS / 'chexpert_images.zip',
    'nih': ZIPS / 'nih_images.zip',
}

for site, path in ARCHIVES.items():
    status = f'{path.stat().st_size / 1e9:.1f} GB' if path.exists() else 'NOT FOUND'
    print(f'{site:<10} {path.name:<24} {status}')

In [ ]:
# --- copy archives to local scratch ------------------------------------------------
local = {}
for site, path in ARCHIVES.items():
    if not path.exists():
        print(f'{site:<10} skipped (archive not found)')
        continue
    target = SCRATCH / path.name
    if not target.exists():
        print(f'{site:<10} copying {path.name} ...')
        shutil.copy2(path, target)
    local[site] = target
    print(f'{site:<10} ready at {target}')

In [ ]:
# --- build or load image cache ------------------------------------------------------
indices = {}
for site, archive in local.items():
    out_dir = CACHE / site
    spec = cfg.sites[site]
    print(f'\n=== {site} ===')
    print(f"key: depth={spec.get('image_key_depth', 1)} "
          f"pattern={spec.get('image_key_pattern')}")

    if (out_dir / 'manifest.json').exists():
        try:
            indices[site] = load_cache(out_dir, cohort_ids=cohorts[site]['image_id'],
                                       config_hash=cfg.cohort_hash)
            print(f'existing cache matches this cohort: {len(indices[site]):,} images')
            continue
        except Exception as exc:
            print(f'existing cache rejected ({exc}); rebuilding')

    indices[site] = build_cache(
        archive,
        cohorts[site]['image_id'],
        out_dir,
        image_size=cfg.model['image_size'],
        key_depth=int(spec.get('image_key_depth', 1)),
        key_pattern=spec.get('image_key_pattern'),
        site=site,
        config_hash=cfg.cohort_hash,
    )
    print(f"packed {len(indices[site]):,} images into "
          f"{indices[site].manifest['n_shards']} shards")

In [ ]:
# --- verify cache ------------------------------------------------------
for site, index in indices.items():
    report = verify_cache(index, sample=1000, cohort_ids=cohorts[site]['image_id'])
    print(f'=== {site} ===')
    print(report.to_string(index=False))
    assert report.ok.all(), f'{site}: cache verification failed'
    print()

In [ ]:
# --- show some cached images ------------------------------------------------------
import matplotlib.pyplot as plt

from pxr.data.cache import read_images

site = cfg.training_sites[0]
sample = cohorts[site].head(6)
pixels = read_images(indices[site], sample['image_id'])

fig, axes = plt.subplots(1, 6, figsize=(15, 3))
for ax, (_, row), img in zip(axes, sample.iterrows(), pixels):
    ax.imshow(img, cmap='gray')
    ax.set_title(f"{row['view']} {row['sex'][0]}{int(row['age'])}", fontsize=9)
    ax.axis('off')
plt.suptitle(f'{site}: cached images with their cohort metadata')
plt.tight_layout()
plt.show()

In [ ]:
# --- cleanup local scratch ------------------------------------------------------
for site, path in local.items():
    if path.exists():
        path.unlink()
        print(f'{site:<10} removed local copy')

In [ ]:
# --- integrity cell------------------
print(f'config_hash : {cfg.config_hash}')
for site, index in indices.items():
    m = index.manifest
    gb = m['n_images'] * m['image_size'] ** 2 / 1e9
    print(f"  {site:<10} {m['n_images']:>7,} images  {m['n_shards']:>3} shards  "
          f"{gb:.2f} GB  fingerprint {m['cohort_fingerprint']}")
    print(f"key depth={m['key_depth']} pattern={m['key_pattern']}")